In [ ]:
from pathlib import Path

import pyvista as pv

from uacpy.anatomies import left_atrium_4_pvs
from uacpy.common import mesh_utils
from uacpy.construction import check, constructor
from uacpy.output import serialization, visualization

In [ ]:
patient_id = "01"
data_dir = Path(f"../data/patient_{patient_id}")
input_mesh_path = data_dir / "mesh_with_fibers_tags.vtu"
uac_submesh_path = data_dir / "uac_submesh_data.npz"
anatomical_tags = {"MV": 0, "LAA": 1, "LIPV": 2, "LSPV": 3, "RIPV": 4, "RSPV": 5}

mesh = pv.read(input_mesh_path)
mesh = mesh_utils.convert_unstructured_to_polydata_mesh(mesh)

In [ ]:
uac_construction_settings = constructor.UACConstructorSettings(
    mesh=mesh,
    feature_tags=anatomical_tags,
    path_config=left_atrium_4_pvs.path_configs,
    marker_config=left_atrium_4_pvs.marker_configs,
    parameterization_config=left_atrium_4_pvs.parameterization_configs,
    uac_config=left_atrium_4_pvs.uac_configs,
    submesh_config=left_atrium_4_pvs.submesh_configs,
)
uac_constructor = constructor.UACConstructor(uac_construction_settings)
uac_constructor.construct_segmentation()
uac_constructor.construct_uacs()

In [ ]:
visualization.plot_mesh_with_segmentation(
    uac_constructor.data.marker_data,
    uac_constructor.data.uac_path_data,
    mesh,
)
visualization.plot_uac_mesh(uac_constructor.data.uac_submesh_data)
serialization.save_uac_submeshdata(uac_submesh_path, uac_constructor.data.uac_submesh_data)
check.check_uac_mesh_validity(uac_constructor.data.uac_submesh_data)